<a href="https://colab.research.google.com/github/zion2010p/ai/blob/main/ai%EB%B0%98%EB%B3%B5%EA%B0%9C%EC%84%A0%EB%A3%A8%ED%94%84_ipnyb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.5 MB/s eta 0:00:00


In [ ]:
from groq import Groq

# Groq 콘솔에서 무료 발급: https://console.groq.com/keys
API_KEY = ""  # gsk_...

client = Groq(api_key=API_KEY)
print("✅ Groq 클라이언트 준비 완료")

✅ Groq 클라이언트 준비 완료


In [ ]:
import time
import re

# ── 설정 ──────────────────────────
MODEL    = "llama-3.3-70b-versatile"
MAX_ITER = 5
DELAY    = 3
# ──────────────────────────────────

def extract_section(text: str, tag: str) -> str:
    """[TAG] ... 형식에서 내용 추출"""
    match = re.search(rf"\[{tag}\](.*?)(?=\[[A-Z]+\]|$)", text, re.DOTALL)
    return match.group(1).strip() if match else ""


def self_improving_ai(question: str) -> str:
    """
    이전 답변 + 비판 + 개선 내용을 messages에 계속 누적하면서
    AI가 스스로 품질을 높이는 루프
    """

    # ── 컨텍스트 누적 시작 ──────────────────────
    messages = [
        {
            "role": "system",
            "content": """너는 자기 개선형 AI다.
질문에 답변한 뒤 스스로 비판하고 개선된 답변을 만들어라.

반드시 아래 형식을 정확히 지켜라:

[ANSWER]
(현재 답변)

[CRITIQUE]
(현재 답변의 문제점)

[IMPROVED]
(개선된 최종 답변)

[DECISION]
CONTINUE 또는 STOP
(STOP: 답변이 충분히 좋음 / CONTINUE: 아직 개선 필요)"""
        },
        {
            "role": "user",
            "content": question
        }
    ]

    best_answer = ""

    for i in range(MAX_ITER):
        print(f"\n🔁 [반복 {i+1}] AI 응답 생성 중...")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.7
        )
        content = response.choices[0].message.content

        # 각 섹션 파싱
        answer   = extract_section(content, "ANSWER")
        critique = extract_section(content, "CRITIQUE")
        improved = extract_section(content, "IMPROVED")
        decision = extract_section(content, "DECISION")

        print(f"📝 답변: {answer[:60]}...")
        print(f"🔍 비판: {critique[:60]}...")
        print(f"✨ 개선: {improved[:60]}...")
        print(f"🎯 판단: {decision}")

        # 🔥 핵심: improved만 골라서 누적
        best_answer = improved if improved else answer

        messages.append({
            "role": "assistant",
            "content": content
        })

        # 다음 반복에서 이전 개선 내용을 기억하도록 누적
        if "CONTINUE" in decision.upper():
            messages.append({
                "role": "user",
                "content": f"""이전 개선 답변:
{best_answer}

위 답변을 바탕으로 더 발전시켜라.
아까 지적한 문제점: {critique}"""
            })
            print(f"🔄 개선 이력 누적 후 재시도...")
            time.sleep(DELAY)

        elif "STOP" in decision.upper():
            print(f"\n✅ AI가 스스로 종료 결정 (반복 {i+1}회)")
            break

    return best_answer


# ────────────────────────────────
# 실행
# ────────────────────────────────
question = "오늘 점심 뭐 먹을지 추천해줘"

print("=" * 50)
print(f"👤 질문: {question}")
print("=" * 50)

answer = self_improving_ai(question)

print("\n" + "=" * 50)
print("💬 최종 답변:")
print("=" * 50)
print(answer)

👤 질문: 오늘 점심 뭐 먹을지 추천해줘

🔁 [반복 1] AI 응답 생성 중...
📝 답변: 오늘 점심은 한국식 소불고기를 추천합니다. 소불고기는 한국의 대표적인 음식 중 하나로, 고기를 달콤하고 시원...
🔍 비판: 현재 답변은 너무 단순하며, 개인의 취향이나 식사 장소, 시간 등을 고려하지 않습니다. 또한, 소불고기가 모...
✨ 개선: 오늘 점심은你的 위치와 취향에 따라 다양한 옵션을 추천합니다. 만약 당신이 한국 음식을 좋아한다면, 소불고기...
🎯 판단: CONTINUE
개선된 답변은 여전히 개인의 취향이나 위치를 고려하지 않으며, 더 구체적인 정보를 제공하지 않습니다. 따라서, 계속해서 개선할 필요가 있습니다.
🔄 개선 이력 누적 후 재시도...

🔁 [반복 2] AI 응답 생성 중...
📝 답변: 오늘 점심은你的 위치와 취향에 따라 다양한 옵션을 추천합니다. 만약 당신이 한국 음식을 좋아한다면, 소불고기...
🔍 비판: 현재 답변은 여전히 너무 일반적이며, 특정한 개인의 취향이나 상황을 고려하지 않습니다. 또한, 구체적인 음식...
✨ 개선: 오늘 점심은你的 위치, 취향, 식사 시간, 예산 등을 고려하여 다양한 옵션을 추천합니다. 만약 당신이 한국 ...
🎯 판단: CONTINUE
개선된 답변은 여전히 개인의 취향이나 상황을 고려하지 않으며, 더 구체적인 정보를 제공하지 않습니다. 따라서, 계속해서 개선할 필요가 있습니다.
🔄 개선 이력 누적 후 재시도...

🔁 [반복 3] AI 응답 생성 중...
📝 답변: 오늘 점심은你的 위치, 취향, 식사 시간, 예산, 식습관 등 모든 요소를 고려하여 개인화된 추천을 제공합니다...
🔍 비판: 현재 답변은 여전히 개인의 취향이나 상황을 고려하지 않으며, 구체적인 음식점이나 식당을 추천하지 않습니다. ...
✨ 개선: 오늘 점심은你的 위치, 취향, 식사 시간, 예산, 식습관 등 모든 요소를 고려하여 개인화된 추천을 제공합니다...
🎯 판단: STOP
개선된 답변은 개인의 취향이나 상황을 고려하며, 구체적인 